In [1]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
import function_sampling_Thermal_forcing_files as fn
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'


In [2]:
 
mnt_pth = '/Volumes/CrucialX8/computing/data/antarctica/' #mount path
pth_imip6hack= mnt_pth + 'ismip6_hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6_hackathon/ComputedScalarsHelene/'
 
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = [] # specify models to remove
 

# Generate loop_info DataFrame
y2sec=-365.25*24*3600
factor = y2sec/10**12

In [3]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

In [4]:
models=[]
paths =[]
exps =[]
region_names = ['Amundsen','Ross','FilchnerRonne']
dic_sectors ={}
dic_sectors ['Amundsen']=np.array([4])
dic_sectors ['Ross']=np.array([2,6])
dic_sectors ['FilchnerRonne']=np.array([5,11])
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()
    #getting rid of old IMAU file (orig_...) and using the newly computed jb_computed... only:
    lst = lst[:-4]
    lst = lst[::-1]
    lst.sort()
    

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
df['AIS']=0
for i in range(1,4):
    df['region_'+str(i)]=0


for key in dic_sectors.keys():
    df[key] = 0
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
#     pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
#     pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
#     d2 = xr.open_dataset(pth_area,decode_times=False )
#     #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )
    tmin= 285
    #melting per area
    
    y =d.shelfmelt.values[:tmin]* factor #GT/m^3 yr
   

    


    m90times = d.time.values[:tmin][np.nanargmax(y)]

  
   

    df['AIS'].iloc[i]= m90times
#     df2['AIS'].iloc[i]= x[:50].mean()

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
    
        m90times = d.time.values[np.nanargmax(y)]
       
      

  
       
     
        df['sector_'+str(j)].iloc[i] = m90times
        

    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]*factor
       
        m90times = d.time.values[np.nanargmax(y)]
       
      

  
       
     
        df['region_'+str(j)].iloc[i] = m90times
        
        
    for key in dic_sectors.keys():
        bmb =0
     
        for j in dic_sectors[key]:
  
            bmb =bmb +d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
           
       
        y = bmb
        m90times = d.time.values[np.nanargmax(y)]
      
    
        df[key].iloc[i]= m90times
#         df2['sector_'+str(j)].iloc[i] = x[:50].max()
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break


/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_83324/1025982299.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]= m90times
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_83324/1025982299.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = m90times
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_83324/1025982299.py:89: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-

In [5]:
df.to_csv('tables/time_maximum_BMB.csv', index=False)

In [6]:
df[df.Experiment ==exp].Model.values

array(['DC_ISSM', 'DOE_MALI_4km', 'DOE_MALI_8km_Ant95',
       'DOE_MALI_8km_AntMean', 'IGE_ElmerIce', 'ILTS_SICOPOLIS',
       'LSCE_GRISLI2', 'LSCE_GRISLI', 'NCAR_CISM1', 'NCAR_CISM2',
       'NORCE_CISM2-MAR364-ERA-t1', 'NORCE_CISM3-MAR364-ERA-t1-local',
       'NORCE_CISM3-MAR364-ERA-t1-nonlocal', 'NORCE_CISM3-MAR364-ERA-t1',
       'NORCE_CISM4-MAR364-ERA-t1-local',
       'NORCE_CISM4-MAR364-ERA-t1-nonlocal', 'NORCE_CISM4-MAR364-ERA-t1',
       'NORCE_CISM4-MAR364-JRA-t1', 'NORCE_CISM5-MAR364-ERA-t1-local',
       'NORCE_CISM5-MAR364-ERA-t1-nonlocal', 'NORCE_CISM5-MAR364-ERA-t1',
       'PIK_PISM', 'UCM_Yelmo', 'UCSD_ISSM', 'ULB_fETISh-KoriBU1',
       'ULB_fETISh-KoriBU2', 'UNN_Ua', 'UTAS_ElmerIce', 'VUB_AISMPALEO',
       'VUW_PISM1', 'VUW_PISM1_s1', 'VUW_PISM1_s2', 'VUW_PISM1_s3',
       'VUW_PISM1_s4', 'VUW_PISM2', 'VUW_PISM2_s1', 'VUW_PISM2_s2',
       'VUW_PISM2_s3', 'VUW_PISM2_s4', 'IMAU_UFEMISM1', 'IMAU_UFEMISM2',
       'IMAU_UFEMISM3', 'IMAU_UFEMISM4'], dtype=object)

In [20]:
df.iloc[-1]

Experiment                                                 expAE05
Model                                                IMAU_UFEMISM4
Path             /Volumes/CrucialX8/computing/data/antarctica/i...
sector_1                                                      2046
sector_2                                                      2100
sector_3                                                      2031
sector_4                                                      2053
sector_5                                                      2162
sector_6                                                      2103
sector_7                                                      2131
sector_8                                                      2123
sector_9                                                      2134
sector_10                                                     2031
sector_11                                                     2095
sector_12                                                     